In [0]:
%sql


create external volume if not exists deltalake.silver.chekcpoint
location 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/silver/checkpo'
comment 'silverloaction ';

In [0]:
%python
df_bronze = spark.readStream.format("delta") \
    .load("/Volumes/deltalake/bronze/rawinbound")

In [0]:
%python
from pyspark.sql.functions import col

df_clean = (
    df_bronze
    .filter(col("event_id").isNotNull())
    .withColumn("event_time", col("event_time").cast("timestamp"))
    .filter(col("event_time").isNotNull())
)

In [0]:
df_watermarked = df_clean.withWatermark("event_time", "10 minutes")

In [0]:
df_dedup = df_watermarked.dropDuplicates(["event_id"])

In [0]:
from pyspark.sql.functions import current_timestamp

df_enriched = df_dedup.withColumn("processed_time", current_timestamp())

In [0]:
query_silver = (
    df_enriched.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/deltalake/silver/chekcpoint")
    .outputMode("append")
    .toTable("deltalake.silver.orders")
)

In [0]:
%sql
CREATE TABLE deltalake.silver.orders
USING DELTA
LOCATION 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/silver/data';

[RequestId=fed0e114-2f89-48bb-adca-454f9ea5bf65 ErrorClass=INVALID_PARAMETER_VALUE] Unsupported path operation PATH_CREATE_TABLE on volume

In [0]:
%sql
select * from deltalake.silver.orders;
